# US-05 — PDF extraction spike (visual demo)

This notebook is the *eyeball* companion to `tests/ingestion/test_extraction_setup.py`.
It runs the hybrid extraction toolchain on the smallest construction guide
(`PCBUs-Working-Together-GPG.pdf`) so we can see the output before trusting it:

- **PyMuPDF (fitz)** for prose,
- **pdfplumber** for tables.

Run it with the project virtualenv kernel (`.venv`). Paths come from
`src.config.settings`, never hard-coded.

In [1]:
# Make the repo root importable so `src.config` resolves from notebooks/.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import fitz  # PyMuPDF
import pdfplumber
from src.config import settings

print("PyMuPDF", fitz.pymupdf_version)
print("pdfplumber", pdfplumber.__version__)
print("sample PDF exists:", settings.SAMPLE_PDF.exists())

PyMuPDF 1.28.0
pdfplumber 0.11.10
sample PDF exists: True


## 1. PyMuPDF — prose on a single page (page 7)

In [2]:
with fitz.open(str(settings.SAMPLE_PDF)) as doc:
    prose = doc[7].get_text()

print(f"{len(prose)} characters\n")
print(prose[:800])

2793 characters

6
1.0 Understanding your duties under HSWA
What you need to know before reading this guide
If you share duties with other PCBUs, what must you do?
You must consult, cooperate with and coordinate activities with all other  
PCBUs you share duties with, so far as is reasonably practicable.
Where duties are shared, all PCBUs have a responsibility to meet those duties,  
to the extent that they have the ability to influence or control the matter.
If you’re self-employed
If a self-employed person is working for another PCBU (for example, a self-
employed welder who is contracted by a labour hire company), they both  
share duties as a PCBU. If the self-employed person decides how their own  
work is done and creates and controls risks, they are considered to have the 
ability to influence or co


## 2. PyMuPDF — text from every page (not just the first)

In [3]:
with fitz.open(str(settings.SAMPLE_PDF)) as doc:
    pages = [doc[i].get_text() for i in range(doc.page_count)]

non_empty = [i for i, t in enumerate(pages) if t.strip()]
print(f"total pages: {len(pages)}")
print(f"pages with text: {len(non_empty)}")
print(f"char count per page: {[len(t) for t in pages]}")

total pages: 36
pages with text: 36
char count per page: [107, 578, 300, 516, 540, 239, 2103, 2793, 3253, 1262, 3487, 2869, 3129, 2490, 2724, 2714, 2088, 255, 2369, 3077, 2627, 3289, 2759, 2698, 822, 209, 1315, 1550, 1293, 1418, 1691, 1636, 3671, 1922, 1019, 123]


## 4. Full document extraction (the US-05 script)

Runs `extract_all_text` from `src/ingestion/extract.py` — the actual extraction
tool — on the whole document, so we can eyeball that every page produces text.
No cleaning or table reconstruction here; that's US-06 / US-07.

In [5]:
from src.ingestion.extract import extract_all_text

pages = extract_all_text(settings.SAMPLE_PDF)
total_chars = sum(len(text) for text in pages)
print(f"{len(pages)} pages, {total_chars} characters total\n")

for i, text in enumerate(pages):
    print("=" * 70)
    print(f"PAGE {i}  ({len(text.strip())} chars)")
    print("=" * 70)
    print(text.strip() or "[no extractable text — possible image-only page]")
    print()

36 pages, 64935 characters total

PAGE 0  (106 chars)
G O O D P R A C T I C E G U I D E L I N E S
PCBUs  
working  
together
ADVICE WHEN  
CONTRACTING
June 2019

PAGE 1  (577 chars)
ACKNOWLEDGEMENTS
WorkSafe would like to acknowledge and thank the stakeholders 
who have contributed to the development of this guidance.
These guidelines are for Persons Conducting 
a Business or Undertaking (PCBUs) who are 
sharing a workplace with other businesses, 
or are working as part of a contracting chain.
They provide advice on how you can meet 
your duties under the Health and Safety at 
Work Act 2015 (HSWA), illustrate different 
contractual relationships between parties, 
and provide examples of ways you can build 
health and safety into contract management.

PAGE 2  (299 chars)
KEY POINTS
–
–
You must consult, cooperate and coordinate with 
other PCBUs when working in a shared workplace,  
or as part of a contracting chain
–
–
You can’t contract out of health and safety duties
–
–
You should a